In [34]:
# Imports used by preprocessing and the recommendation model.
import ast
import html
import re
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer

In [35]:
# Load the source data and keep products with both feature sources available.
DATA_DIR = Path("data")
df = pd.read_csv(DATA_DIR / "product_info.csv")

products = (
    df.dropna(subset=["highlights", "ingredients"])
    .copy()
    .reset_index(drop=True)
)

In [36]:
# convert str of list into python list
def parse_highlights(value):
    
    
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return parsed
    except (ValueError, SyntaxError):
        pass

    return [str(value)]


products["highlights_list"] = products["highlights"].apply(
    parse_highlights
)

In [37]:
def clean_highlight(text):
    # convert to lowercase
    text = str(text).lower().strip()

    # normalize unicode characters
    text = unicodedata.normalize("NFKC", text)

    # replace hyphens with spaces
    text = re.sub(r"[-‐‑–—]", " ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text)

    return text

products["cleaned_highlights"] = (
    products["highlights_list"].apply(
        lambda items: [clean_highlight(h) for h in items]
    )
)

In [38]:
#  now splite "good for: A,B,C" into "good for A, good for B ..."
skin_type_mapping = {
    "best for dry skin": ["skin_dry"],
    "best for oily skin": ["skin_oily"],
    "best for normal skin": ["skin_normal"],
    "best for combination skin": ["skin_combination"],
    "best for dry, combo, normal skin": [
        "skin_dry", "skin_combination", "skin_normal"
    ],
    "best for oily, combo, normal skin": [
        "skin_oily", "skin_combination", "skin_normal"
    ]
}

#  revomoving hihglights non-related
marketing_highlights = {
    "community favorite",
    "clean at sephora",
    "clean + planet positive",
    "black owned at sephora"
}

def normalize_highlight(highlight):
    # Normalize text formatting
    h = highlight.lower().strip()
    h = h.replace("-", " ")
    h = re.sub(r"\s+", " ", h)

    # Remove marketing features from similarity
    if h in marketing_highlights or h.startswith("allure "):
        return []

    # Expand combined skin-type labels
    if h in skin_type_mapping:
        return skin_type_mapping[h]

    # Normalize remaining highlights
    return [h.replace(" ", "_")]


    
def preprocess_highlights(highlights):
    normalized = []

    for highlight in highlights:
        normalized.extend(normalize_highlight(highlight))

    # Remove duplicates while preserving order
    return list(dict.fromkeys(normalized))


products["normalized_highlights"] = (
    products["cleaned_highlights"].apply(preprocess_highlights)
)

In [ ]:
def normalize_ingredient(name):
    
    # clean some formatiing 
    if not isinstance(name, str):
        return ""

    #  decode some html values
    name = html.unescape(name)
    name = unicodedata.normalize("NFKC", name).casefold()

    # replace all possible dashes with one type
    name = re.sub(r"[‐‑‒–—−]", "-", name)

    # remove extra white spaces
    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"\s*/\s*", "/", name)

    name = re.sub(r"[\s*†‡]+$", "", name)
    # removes speces, periods, semicolms
    return name.strip(" .;")

# splite the ingridients, while being mindful of the brakets in notation
# "Water, Example Extract (leaf, stem), Glycerin" -> ["water", "example extract (leaf, stem)", "glycerin"]
# Making it numbers safe so we dont acesently splite. anumber into differnt ingridients
def split_ingredient_text(text):
    #  to store the names of ingridients
    ingredients = []
    
    # the current chars of the ingridient
    current = []
    
    # opened parathethis 
    stack = []

    # 1) find numeric prefixes such as "1,2-" or "1,2,3-".
    # 2) save the positions of their commas so they are not treated
    # as separators between ingredients.
    numeric_commas = set()

    numeric_prefix_pattern = r"(?<!\w)\d+(?:,\d+)+\s*[-‐‑‒–—−]"

    for match in re.finditer(numeric_prefix_pattern, text):
        for position in range(match.start(), match.end()):
            if text[position] == ",":
                numeric_commas.add(position)

    # read the ingredient text one character at a time.
    for position, char in enumerate(text):

        if char in "([":
            stack.append(char)

        elif char in ")]":
            expected = "(" if char == ")" else "["

            if not stack or stack[-1] != expected:
                raise ValueError(
                    "Unbalanced ingredient parentheses/brackets"
                )

            stack.pop()

        # Aa comma separates ingredients only if it is:
        # 1)outside parentheses and brackets and
        # 2)not part of a numeric prefix
        is_separator = (
            char == ","
            and not stack
            and position not in numeric_commas
        )

        if is_separator:
            ingredient = normalize_ingredient(
                "".join(current)
            )

            if ingredient:
                ingredients.append(ingredient)

            # starting collecting the next ingredient.
            current = []

        else:
            current.append(char)

    # An opening parenthesis or bracket was never closed.
    if stack:
        raise ValueError(
            "Unbalanced ingredient parentheses/brackets"
        )

    # Add the final ingredient, which has no comma after it.
    ingredient = normalize_ingredient(
        "".join(current)
    )

    if ingredient:
        ingredients.append(ingredient)

    return ingredients


# also assignes a stustues to an ingridient list
# missing -> no values
# invalid-list - > invalid python list
# invalid section - contains elem that is not a sring
# needs_section_review - contains more than one section of ingridients, like seperate part ususlly devided by : 
# needs_section_review - missing brakets
# empty
def parse_ingredients(value):
    if isinstance(value, list):
        sections = value

    elif isinstance(value, str):
        value = value.strip()

        if not value:
            return {"ingredients": [], "status": "missing"}

        if value.startswith("["):
            try:
                # converts text into actual list python
                sections = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return {"ingredients": [], "status": "invalid_list"}
        else:
            sections = [value]

    else:
        return {"ingredients": [], "status": "missing"}

    if not isinstance(sections, list):
        return {"ingredients": [], "status": "invalid_list"}

    if not all(isinstance(section, str) for section in sections):
        return {"ingredients": [], "status": "invalid_section"}

    sections = [section.strip() for section in sections if section.strip()]

    if not sections:
        return {"ingredients": [], "status": "missing"}

    if len(sections) > 1:
        return {"ingredients": [], "status": "needs_section_review"}

    try:
        ingredients = split_ingredient_text(sections[0])
    except ValueError:
        return {"ingredients": [], "status": "needs_delimiter_review"}

    return {
        "ingredients": ingredients,
        "status": "parsed" if ingredients else "empty",
    }

In [40]:
APPROVED_CORRECTIONS = {
    "hydroxyacetophe-none": "hydroxyacetophenone",
    "pan-thenol": "panthenol",
}

APPROVED_ALIASES = {
    "aqua/water/eau": "water",
    "water/aqua/eau": "water",
    "aqua (water)": "water",
    "water (aqua)": "water",
}

CANONICAL_MAP = {
    normalize_ingredient(source): normalize_ingredient(target)
    for source, target in {
        **APPROVED_CORRECTIONS,
        **APPROVED_ALIASES,
    }.items()
}


def canonicalize_ingredients(ingredients):
    # applying the approved replacement
    cleaned = []

    for ingredient in ingredients:
        name = normalize_ingredient(ingredient)
        name = CANONICAL_MAP.get(name, name)

        if name:
            cleaned.append(name)

    return list(dict.fromkeys(cleaned))

In [41]:
parsed = pd.DataFrame(
    products["ingredients"].apply(parse_ingredients).tolist(),
    index=products.index,
)

products["ingredients_list"] = parsed["ingredients"]
products["ingredient_parse_status"] = parsed["status"]


products["normalized_ingredients"] = (
    products["ingredients_list"].apply(canonicalize_ingredients)
)

print(products["ingredient_parse_status"].value_counts())

# recreate skincare after updating products.
skincare = products.loc[
    products["primary_category"].eq("Skincare")
].copy()

<unknown>:1: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.


ingredient_parse_status
parsed                    4987
needs_section_review       798
needs_delimiter_review      35
Name: count, dtype: int64


In [42]:
# want to find cheaper per ouces, for the better comparison will conver everyhting to ml
import re
import numpy as np
import pandas as pd


# get product size
def parse_product_size(size):
    if not isinstance(size, str):
        return np.nan, None

    size = size.lower().strip()

    # Reject multipacks that need special handling
    if re.search(r'\d+\s*[x×]\s*\d', size):
        return np.nan, None

    # Prefer explicitly stated metric units
    matches = re.findall(
        r'(?<!\w)(\d*\.?\d+)\s*(ml|milliliters?|grams?|g)\b',
        size
    )

    if matches:
        quantities = [
            (
                float(value),
                "mL" if unit.startswith(("ml", "milliliter")) else "g"
            )
            for value, unit in matches
        ]

        # Avoid choosing a size when multiple quantities conflict
        if len(set(quantities)) != 1:
            return np.nan, None

        value, unit = quantities[0]
        return (value, unit) if value > 0 else (np.nan, None)

    # Convert explicitly labeled fluid ounces to milliliters
    match = re.search(
        r'(?<!\w)(\d*\.?\d+)\s*fl\.?\s*oz\b',
        size
    )
    # Handle oz without an explicit fluid/weight designation
    match = re.search(
        r'(?<!\w)(\d*\.?\d+)\s*oz\b',
        size
    )

    if match:
        value = float(match.group(1))
        return value, "oz"

    if match:
        value_ml = float(match.group(1)) * 29.5735
        return (value_ml, "mL") if value_ml > 0 else (np.nan, None)

    # Ambiguous ounces, sets, or unknown units
    return np.nan, None

In [43]:
# Keep individual skincare formulas.
eligible_categories = {
    "Cleansers",
    "Eye Care",
    "Lip Balms & Treatments",
    "Masks",
    "Moisturizers",
    "Self Tanners",
    "Sunscreen",
    "Treatments",
}


# working only with skincare from chosen ligible_categories and parsed, so dont deal with complicated stuff
skincare_model = products.loc[
    products["primary_category"].eq("Skincare")
    & products["secondary_category"].isin(eligible_categories)
    & products["ingredient_parse_status"].eq("parsed")
    & products["normalized_ingredients"].map(bool)
].copy()

skincare_model["price_usd"] = pd.to_numeric(
    skincare_model["price_usd"],
    errors="coerce",
)

skincare_model = (
    skincare_model
    .drop_duplicates(subset="product_id")
    .reset_index(drop=True)
)

if skincare_model.empty:
    raise ValueError("No eligible products. Check the preprocessing cells.")

print("Products available:", len(skincare_model))

Products available: 1525


In [44]:
# Extract size and unit
skincare_model[["size_value", "size_unit"]] = pd.DataFrame(
    skincare_model["size"].apply(parse_product_size).tolist(),
    index=skincare_model.index,
)

# Calculate unit price
skincare_model["price_per_unit"] = (
    skincare_model["price_usd"] / skincare_model["size_value"]
)

# Inspect the results
display(
    skincare_model[
        [
            "product_name",
            "size",
            "size_value",
            "size_unit",
            "price_usd",
            "price_per_unit",
        ]
    ].head(20)
)

,product_name,size,size_value,size_unit,price_usd,price_per_unit
0,GENIUS Sleeping Collagen Moisturizer,2 oz/ 60 mL,60.0,mL,98.0,1.633333
1,GENIUS Liquid Collagen Serum,1 oz / 30 mL,30.0,mL,115.0,3.833333
2,GENIUS Liquid Collagen Lip Treatment,.5 oz / 15 mL,15.0,mL,29.0,1.933333
3,SUBLIME DEFENSE Ultra Lightweight UV Defense F...,1 oz,1.0,oz,28.0,28.000000
4,GENIUS Ultimate Anti-Aging Cream,2 oz/ 60 mL,60.0,mL,112.0,1.866667
5,GENIUS Ultimate Anti-Aging Vitamin C+ Serum,1 oz/ 30 mL,30.0,mL,118.0,3.933333
6,Algae Niacinamide Moisture Veil,1.7 oz/ 50 mL,50.0,mL,65.0,1.300000
7,GENIUS Liquid Skin Resurfacing 2% BHA Toner,3.4 oz/ 100 mL,100.0,mL,40.0,0.400000
8,GENIUS Collagen Calming Relief,1.35 oz/ 40 mL,40.0,mL,58.0,1.450000
9,ALIVE Prebiotic Balancing Mask,1.7 oz/ 50 mL,50.0,mL,38.0,0.760000


In [45]:
import numpy as np
from sklearn.preprocessing import normalize


# make rows aligned with both matrices.
skincare_model = skincare_model.reset_index(drop=True)

# initialised the Tf-idf using ingridients
ingredient_vectorizer = TfidfVectorizer(
    analyzer=lambda ingredients: ingredients,
    lowercase=False,
)

# actully apply the vectorised and assign frequencies to the ingridients
X_ingredients = ingredient_vectorizer.fit_transform(
    skincare_model["normalized_ingredients"]
)

# we want our ingridient list to be sensitive to the positioning/order
# decay - how fast the weight destreas, the greatwe -> slower decrease
def position_weight(position, decay=8, min_weight=0.3):
    return min_weight + (1 - min_weight) * np.exp(
        -position / decay
    )

# Preserve original TF-IDF matrix
X_ingredients_original = X_ingredients.copy()

# Create position-weighted TF-IDF matrix
X_ingredients_position = X_ingredients.copy().tolil()

# Map ingredient name to its TF-IDF column
vocab = ingredient_vectorizer.vocabulary_

for row_idx, ingredients in enumerate(
    skincare_model["normalized_ingredients"]
):
    for position, ingredient in enumerate(ingredients):

        col_idx = vocab[ingredient]

        weight = position_weight(position)

        # now actully adjusting the TF-IDF score based on position
        X_ingredients_position[row_idx, col_idx] *= weight

# Convert back to efficient sparse matrix format
X_ingredients_position = normalize(
    X_ingredients_position.tocsr(),
    norm="l2"
)

print("Position-weighted matrix:", X_ingredients_position.shape)




# Now highlights
# each highlight becomes a binary feature: present or absent
highlight_encoder = MultiLabelBinarizer(sparse_output=True)

X_highlights = highlight_encoder.fit_transform(
    skincare_model["normalized_highlights"]
)


# makes a dictionary products id: row number
product_positions = {
    product_id: position
    for position, product_id in enumerate(skincare_model["product_id"])
}

print("Ingredient feature matrix:", X_ingredients.shape)
print("Highlight feature matrix:", X_highlights.shape)

Position-weighted matrix: (1525, 5156)
Ingredient feature matrix: (1525, 5156)
Highlight feature matrix: (1525, 67)


In [46]:
def recommend_dupes(
    product_id,
    top_k=5,
    # given the ingridients the weight of 70% theofore the highlights are weighted at 30%
    ingredient_weight=0.7,
    different_brand=False,
    position_importance=0.4,
):
    if product_id not in product_positions:
        raise ValueError("Product ID is not in skincare_model.")

    if not isinstance(top_k, int) or top_k < 1:
        raise ValueError("top_k must be a positive integer.")

    if not 0 < ingredient_weight < 1:
        raise ValueError("ingredient_weight must be between 0 and 1.")

    position = product_positions[product_id]
    
    # what are we looking an alternative for
    query = skincare_model.iloc[position]


    # price validation usin gprice per unit
    if (pd.isna(query["price_per_unit"]) or not np.isfinite(query["price_per_unit"]) or query["price_per_unit"] <= 0):
        raise ValueError(
            "The selected product needs a valid price and size."
        )


    if X_highlights[position].nnz == 0:
        raise ValueError(
            "This product has no usable highlights to compare."
        )


    # original TF-IDF similarity (ignores ordering), using kust cosine
    original_scores = cosine_similarity(
        X_ingredients_original[position],
        X_ingredients_original,
    ).ravel()

    # position-aware TF-IDF similarity
    position_scores = cosine_similarity(
        X_ingredients_position[position],
        X_ingredients_position,
    ).ravel()

    # blend both measures, can be changed
    # position_importance defined in the fucntion call

    ingredient_scores = (
        (1 - position_importance) * original_scores
        + position_importance * position_scores
    )

    highlight_scores = cosine_similarity(
        X_highlights[position],
        X_highlights,
    ).ravel()

    # starting weights: 70% ingredients and 30% highlights by default, got these numbers out of head
    combined_scores = (
        ingredient_weight * ingredient_scores
        + (1 - ingredient_weight) * highlight_scores
    )

    # only looking at the comparable product categories
    mask = skincare_model["secondary_category"].eq(
        query["secondary_category"]
    )

    if pd.notna(query["tertiary_category"]):
        mask &= skincare_model["tertiary_category"].eq(
            query["tertiary_category"]
        )
    else:
        mask &= skincare_model["tertiary_category"].isna()

    # alternatives MUST have a lower package price
    # Compare only products with the same measurement unit
    mask &= skincare_model["size_unit"].eq(query["size_unit"])

    # Alternatives MUST have a lower price per unit
    mask &= (
        skincare_model["price_per_unit"].gt(0)
        & skincare_model["price_per_unit"].lt(query["price_per_unit"])
    )

    mask &= skincare_model["product_id"].ne(product_id)

    if different_brand:
        mask &= skincare_model["brand_name"].ne(query["brand_name"])

    # require some overlap in BOTH ingredients and highlights
    mask &= (ingredient_scores > 0) & (highlight_scores > 0)

    results = skincare_model.loc[mask].copy()
    selected = mask.to_numpy()

    results["ingredient_similarity"] = ingredient_scores[selected]
    results["highlight_similarity"] = highlight_scores[selected]
    results["combined_similarity"] = combined_scores[selected]

    results = results.sort_values(
        ["combined_similarity", "price_per_unit", "product_id"],
        ascending=[False, True, True],
    ).head(top_k).copy()

    # explain what each alternative shares with the selected product
    query_ingredients = set(query["normalized_ingredients"])
    query_highlights = set(query["normalized_highlights"])

    results["shared_ingredients"] = results[
        "normalized_ingredients"
    ].apply(lambda items: sorted(query_ingredients & set(items)))

    results["shared_highlights"] = results[
        "normalized_highlights"
    ].apply(lambda items: sorted(query_highlights & set(items)))

    results["package_savings_usd"] = (
        query["price_usd"] - results["price_usd"]
    )

    results["package_savings_pct"] = (
        100 * results["package_savings_usd"] / query["price_usd"]
    )
    
    results["unit_savings_usd"] = (
    query["price_per_unit"] - results["price_per_unit"])

    results["unit_savings_pct"] = (
        100 * results["unit_savings_usd"] / query["price_per_unit"]
    )
    
    

    columns = [
        "product_id",
        "product_name",
        "brand_name",
        "size",
        "price_usd",
        "ingredient_similarity",
        "highlight_similarity",
        "combined_similarity",
        "package_savings_usd",
        "package_savings_pct",
        "shared_highlights",
        "shared_ingredients",
    ]

    return results[columns].reset_index(drop=True)

In [47]:
# HERE put the product id you are looking dupes for 
# for example select the first product with usable highlights and a valid price
example_products = skincare_model.loc[
    skincare_model["normalized_highlights"].map(bool)
    & skincare_model["price_usd"].gt(0)
]




if example_products.empty:
    raise ValueError("No products have both usable highlights and a valid price.")

# assign query_id to the product if you are looking a dupe for 
query_id = example_products.iloc[0]["product_id"]

display(
    skincare_model.loc[
        skincare_model["product_id"].eq(query_id),
        [
            "product_id", "product_name", "brand_name", "size",
            "price_usd", "normalized_highlights",
        ],
    ]
)

# Every returned alternative must be cheaper and share ingredients and highlights.
recommendations = recommend_dupes(
    product_id=query_id,
    top_k=5,
    ingredient_weight=0.7,
    different_brand=False,
)

display(
    recommendations.round({
        "ingredient_similarity": 3,
        "highlight_similarity": 3,
        "combined_similarity": 3,
        "package_savings_usd": 2,
        "package_savings_pct": 1,
    })
)

if recommendations.empty:
    print("No cheaper alternatives match both ingredients and highlights.")

# Savings compare package prices, not equal volumes or weights.
# Shared ingredients do not establish equal concentrations or performance.

,product_id,product_name,brand_name,size,price_usd,normalized_highlights
0,P439055,GENIUS Sleeping Collagen Moisturizer,Algenist,2 oz/ 60 mL,98.0,"[vegan, good_for:_loss_of_firmness, collagen, ..."


,product_id,product_name,brand_name,size,price_usd,ingredient_similarity,highlight_similarity,combined_similarity,package_savings_usd,package_savings_pct,shared_highlights,shared_ingredients
0,P453818,GENIUS Collagen Calming Relief,Algenist,1.35 oz/ 40 mL,58.0,0.226,0.802,0.399,40.0,40.8,"[collagen, hypoallergenic, skin_combination, s...","[caprylyl glycol, cetearyl alcohol, chlorella ..."
1,P411403,Confidence in a Cream Anti-Aging Hydrating Moi...,IT Cosmetics,0.51 oz / 15 mL,20.0,0.116,0.750,0.306,78.0,79.6,"[good_for:_loss_of_firmness, skin_combination,...","[caprylic/capric triglyceride, caprylyl glycol..."
2,P427641,Mini Confidence in a Cream Hydrating Moisturizer,IT Cosmetics,0.51 oz / 15 mL,20.0,0.116,0.750,0.306,78.0,79.6,"[good_for:_loss_of_firmness, skin_combination,...","[caprylic/capric triglyceride, caprylyl glycol..."
3,P483698,Nourishing Squalane Daily Moisturizer,The Outset,1.7 oz / 50 mL,44.0,0.237,0.316,0.261,54.0,55.1,"[hypoallergenic, vegan]","[caprylic/capric triglyceride, caprylyl glycol..."
4,P405599,SEA drink of H2O hydrating boost moisturizer,tarte,1.70 oz/ 50 mL,39.0,0.094,0.625,0.253,59.0,60.2,"[good_for:_loss_of_firmness, skin_combination,...",[ammonium acryloyldimethyltaurate/vp copolymer...
